In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
from models.unet_3d import UNet3D
from utils.unets_helper_functions import (show_organ_overlay,
                    show_difference_map,
                    sliding_window_inference)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet3D().to(device)

model_path = "../experiments/attention_unet_fold0/best_model.pth"
state_dict = torch.load(model_path, map_location=device)

model.load_state_dict(state_dict)
model.eval()

print("Model loaded for testing.")


In [ ]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

In [ ]:
case = val_cases[1]  # take second validation case
print("Testing case:", case)

image = nib.load(f"../data/processed/imagesTr/{case}.nii.gz").get_fdata()
label = nib.load(f"../data/processed/labelsTr/{case}.nii.gz").get_fdata()


In [ ]:
pred = sliding_window_inference(
    model,
    image,
    patch_size=80,
    stride=40,
    device=device
)

print("Prediction shape:", pred.shape)


In [ ]:
z = image.shape[0] // 2

ct_slice = image[z]
gt_slice = label[z]
pred_slice = pred[z]

In [ ]:
plt.figure(figsize=(15,5))

# CT
plt.subplot(1,3,1)
plt.imshow(ct_slice, cmap='gray')
plt.title("CT Slice")
plt.axis("off")

# Ground Truth
plt.subplot(1,3,2)
plt.imshow(ct_slice, cmap='gray')
plt.imshow(gt_slice, alpha=0.5)
plt.title("Ground Truth Mask")
plt.axis("off")

# Prediction
plt.subplot(1,3,3)
plt.imshow(ct_slice, cmap='gray')
plt.imshow(pred_slice, alpha=0.5)
plt.title("Predicted Mask")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
show_difference_map(ct_slice, gt_slice, pred_slice)

In [ ]:
show_organ_overlay(ct_slice, gt_slice, pred_slice)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def interactive_slice_viewer(volume, title="Volume"):

    def view_slice(z):
        plt.figure(figsize=(6,6))
        plt.imshow(volume[z], cmap='gray')
        plt.title(f"{title} - Slice {z}")
        plt.axis("off")
        plt.show()

    slider = widgets.IntSlider(
        min=0,
        max=volume.shape[0]-1,
        step=1,
        value=volume.shape[0]//2
    )

    widgets.interact(view_slice, z=slider)


In [ ]:
interactive_slice_viewer(image, title="CT")

In [ ]:
interactive_slice_viewer(pred, title="Prediction")

In [ ]:
interactive_slice_viewer(label, title="Ground Truth")